In [3]:

import torch
import torchvision
import matplotlib.pyplot as plt
import random
from torchvision.transforms import ToTensor
import numpy as np
import torch.nn as nn

import torch.optim as optim

traindataset=torchvision.datasets.CIFAR10( root="./traindata",
                                      train=True,
                                          download=False,transform=ToTensor())

testdataset=torchvision.datasets.CIFAR10( root="./traindata",
                                      train=False,
                                          download=False,transform=ToTensor())





RuntimeError: Dataset not found or corrupted. You can use download=True to download it

In [ ]:
image,lable=random.choice(traindataset)



image=image.permute(1,2,0)
plt.imshow(image)
plt.title(traindataset.classes[lable])
plt.show()



In [ ]:
class Residualblock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        residual = self.shortcut(x)
        out += residual
        out = self.relu(out) # Standard ResNet applies ReLU after the shortcut addition
        return out

class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_channels = 32

        self.stem = nn.Sequential(
            nn.Conv2d(3, self.in_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(self.in_channels),
            nn.ReLU(inplace=True)
        )

        # Defining stages
        self.layer1 = self.make_stage(out_channels=64, numblock=3, stride=1)
        self.layer2 = self.make_stage(out_channels=128, numblock=3, stride=2)
        self.layer3 = self.make_stage(out_channels=256, numblock=3, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)

    def make_stage(self, out_channels, numblock, stride):
        strides = [stride] + [1] * (numblock - 1)
        layers = []
        for s in strides:
            # FIX: Use dynamic 's' and update self.in_channels for the next block
            layers.append(Residualblock(self.in_channels, out_channels, stride=s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x
def compute_accuracy(outputs, labels):
    _, preds = torch.max(outputs, 1)
    return (preds == labels).sum().item() / len(labels)

In [ ]:
torch.manual_seed(42)
from torch.utils.data import DataLoader
from tqdm import tqdm
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet().to(device)

# FIX: Define loss function and optimizer OUTSIDE the loops
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10

# (Ensure your 'traindataset' is instantiated before this line)
dataloader = DataLoader(traindataset, batch_size=32, shuffle=True)
dataloadertest = DataLoader(testdataset, batch_size=32, shuffle=True)
# --- Training Loop ---
for epoch in tqdm(range(epochs)):

    model.train()
    running_loss, running_acc = 0.0, 0.0
    for batch, (images, labels) in enumerate(dataloader):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels) # FIX: Avoided shadowing variable name

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        acc = compute_accuracy(outputs, labels)
        running_loss += loss.item()
        running_acc += acc


    epoch_loss = running_loss / len(dataloader)
    epoch_acc = running_acc / len(dataloader)
    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_acc)

    model.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
         for batch,(images, labels) in enumerate(dataloadertest):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            trainloss = criterion(outputs, labels)
            val_loss += loss.item()
            val_acc += compute_accuracy(outputs, labels)

    val_loss /= len(dataloadertest)
    val_acc /= len(dataloadertest)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)


In [ ]:
epochs_range = range(1, epochs + 1)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], label='Train Loss', marker='o')
plt.plot(epochs_range, history['val_loss'], label='Val Loss', marker='o')
plt.title('Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, [a*100 for a in history['train_acc']], label='Train Acc', marker='o')
plt.plot(epochs_range, [a*100 for a in history['val_acc']], label='Val Acc', marker='o')
plt.title('Accuracy Curves')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(history['train_loss'])
print(history['train_acc'])
print(history['val_loss'])
print(history['val_acc'])
model.state_dict

In [ ]:
SAVE_PATH = "my_model_weights_const.pth"

# 2. Save the state dictionary
torch.save(model.state_dict(), SAVE_PATH)
print(f"Model weights successfully saved to {SAVE_PATH}")




def save_model_outputs(model, dataloader, device, filepath="test_outputs.pt"):
    """Runs a clean inference pass to gather and save all raw model outputs (logits)

    and true labels from a dataset.
    """
    model.eval()
    all_outputs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)

            # Move data back to CPU to save memory before storing
            all_outputs.append(outputs.cpu())
            all_labels.append(labels.cpu())

    # Concatenate list of batches into single tensors
    output_dict = {
        'outputs': torch.cat(all_outputs, dim=0),
        'true_labels': torch.cat(all_labels, dim=0)
    }

    torch.save(output_dict, filepath)

eval_dataloader = DataLoader(testdataset, batch_size=32, shuffle=False)
save_model_outputs(model, eval_dataloader, device, filepath="final_test_outputs.pt")



In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in dataloadertest:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
classes=traindataset.classes
# Final Accuracy Calculation
final_accuracy = (np.array(all_preds) == np.array(all_labels)).sum() / len(all_labels)
print(f"\n=========================================\nFINAL ACCURACY ON TEST SET: {final_accuracy*100:.2f}%\n=========================================\n")

# Classification Report
print("CLASSIFICATION REPORT:")
print(classification_report(all_labels, all_preds, target_names=classes))

# --- 3. CONFUSION MATRIX ---
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
print(all_preds)
print(all_labels)